# Imports

In [33]:
import json
import os
import shutil
import subprocess


from ase import Atoms
from ase.build import fcc100, molecule
from ase.calculators.emt import EMT
from ase.io import read, write
from ase.visualize import view

import numpy as np
import random

from symmetry_analysis.create import *
from symmetry_analysis.utility import *
from symmetry_analysis.transformations import *
from symmetry_analysis.symmetry import *
from symmetry_analysis.visualization import *
from symmetry_analysis.symmetry_with_sofi import *

from boss.bo.bo_main import BOMain
from boss.pp.pp_main import PPMain

# Obtaining symmetrically equivalent adsorption configurations

## Defining materials

In [35]:
wrkdir = os.getcwd()

In [36]:
au = fcc100('Au', size=(2, 2, 4), vacuum = 10)
water = molecule('H2O')

In [37]:
slab = au.copy()
mol = water.copy()

calculator = EMT()

slab.calc = calculator
mol.calc = calculator

e_slab = slab.get_potential_energy()
e_mol = mol.get_potential_energy()

In [38]:
idx = [0, 1, 2]

## Defining adsorption configurations

In [39]:
slab = au.copy()
mol = water.copy()

# initiating position variables
x = random.uniform(0,0.5)
y = random.uniform(0,0.5)
z = 2.8

# initiating orientation variables
a = random.randint(0, 360)
b = random.randint(0, 360)
c = random.randint(0, 360)

pos_list = [x, y, z]
angle_list = [a, b, c]

# initiating adsorption configuration
ads = create(slab, mol, a, b, c, x, y, z)
ads.calc = calculator

e_system = ads.get_potential_energy()
e_ads = e_system - e_slab - e_mol

print(f'The adsorption energy of the initial configuration is {e_ads}')

The adsorption energy of the initial configuration is 0.7284985038885248


In [40]:
unique, duplicate, out_of_bounds = get_symmetric_positions_and_angles_with_SOFI(
    pos_list,
    angle_list,
    slab,
    mol,
    idx,
    x_bounds=[0, 0.5],
    y_bounds=[0, 0.5],
)

In [41]:
unique_df = dataframe_from_s_prime(unique, slab, mol, ads)
print(unique_df[["Rotation Determinant", 'Adsorption Energy (eV)', 'Is it equivalent?']])

    Rotation Determinant  Adsorption Energy (eV)  Is it equivalent?
0                    1.0                  0.7285               True
1                    1.0                  0.7285               True
2                    1.0                  0.7285               True
3                    1.0                  0.7285               True
4                    1.0                  0.7285               True
5                    1.0                  0.7285               True
6                    1.0                  0.7285               True
7                    1.0                  0.7285               True
8                   -1.0                  0.7285               True
9                   -1.0                  0.7285               True
10                  -1.0                  0.7285               True
11                  -1.0                  0.7285               True
12                  -1.0                  0.7285               True
13                  -1.0                  0.7285

# Using in BOSS

In [42]:
def get_position_unique(data:list, n:int):
    # randomize order
    random.seed(42)
    random.shuffle(data)

    unique_positions = {}
    present_idx = []
    remains = []
    
    for i, s in enumerate(data):
        pos = tuple(s[0])
        if pos not in unique_positions:
            unique_positions[pos] = s
            present_idx.append(i)
    
    for i, s in enumerate(data):
        if i not in present_idx:
            remains.append(s)

    unique_s = list(unique_positions.values())

    if n <= len(unique_s):
        solutions = random.sample(unique_s, n)
        return(solutions)

    else:
        solutions = unique_s
        to_add = n-len(unique_s)
        if to_add < len(remains):
            solutions.extend(random.sample(remains, n-len(unique_s)))
        else:
            solutions.extend(remains)
        return(solutions)

In [43]:
au = fcc100('Au', size=(2, 2, 4), vacuum = 10)
water = molecule('H2O')

In [44]:
slab = au.copy()
mol = water.copy()
n_subsample = 4

bounds = np.array(
    [
        [0, 360],
        [0, 360],
        [0, 360],
        [0, 0.5],
        [0, 0.5],
        [2, 3]
    ]
)

kernels = np.array(
    [
        'stdp',
        'stdp',
        'stdp',
        'stdp',
        'stdp',
        'rbf'
    ]
)

In [45]:
calculator = EMT()

slab.calc = calculator
mol.calc = calculator
e_slab = slab.get_potential_energy()
e_mol = mol.get_potential_energy()

In [46]:
def func(x):
    x = np.squeeze(x)
    alpha, beta, gamma, a, b, c = x
    pos = [a, b, c]
    angle = [alpha, beta, gamma]

    ads = create(slab, mol, alpha, beta, gamma, a, b, c)
    ads.calc = calculator

    e_system = ads.get_potential_energy()
    e_ads = e_system - e_slab - e_mol

    if e_ads > 1:
        e_ads = np.log10(e_ads) + 1

    unique, _, _ = get_symmetric_positions_and_angles_with_SOFI(
        pos,
        angle,
        slab,
        mol,
        idx,
        x_bounds=[0, 0.5],
        y_bounds=[0, 0.5],
    )

    unique_subsampled = get_position_unique(unique, n_subsample)
    X_list = [np.array([*u[1], *u[0]]) for u in unique_subsampled]
    
    X = np.array(X_list)
    Y = np.ones(len(X_list)) * e_ads

    return X, Y

In [47]:
bo = BOMain(
    func,
    bounds,
    acqfn_name = 'exploit',
    kernel=kernels,
    initpts=10,
    iterpts=10
)

In [48]:
res = bo.run()

In [49]:
print('Predicted global min: ', res.select('mu_glmin', -1))

Predicted global min:  -0.2302830211260094
